In [0]:
from pyspark.sql.functions import col, dense_rank
from pyspark.sql.window import Window

def get_top_3_employees(df):
    window_spec = Window.partitionBy("department").orderBy(col("salary").desc())
    
    return (
        df.withColumn("rank", dense_rank().over(window_spec))
          .filter(col("rank") <= 3)
          .drop("rank")
    )

In [0]:
from pyspark.sql import SparkSession

# Test Function
def test_top_3_employees():
    data = [
        (1, "A", "HR", 100),
        (2, "B", "HR", 200),
        (3, "C", "HR", 150),
        (4, "D", "HR", 120),
        (5, "E", "IT", 300),
        (6, "F", "IT", 250),
        (7, "G", "IT", 200),
        (8, "H", "IT", 100),
    ]

    columns = ["emp_id", "emp_name", "department", "salary"]
    df = spark.createDataFrame(data, columns)

    result_df = get_top_3_employees(df)

    # Convert result to set (order-independent comparison)
    result = set((row.emp_name, row.department) for row in result_df.collect())

    expected = {
        ("B", "HR"), ("C", "HR"), ("D", "HR"),
        ("E", "IT"), ("F", "IT"), ("G", "IT")
    }

    # Assertion
    assert result == expected, f"Test Failed: Expected {expected}, Got {result}"

    print("✅ Test Passed!")

# Run test
test_top_3_employees()